# 01 データ読込・前処理 (再作成)
分割ノート再生成。後続ノート (02,03,04) が利用する中間ファイルを parquet 化して保存します。

In [ ]:
import pandas as pd, os, jpholiday
from datetime import datetime, date, timedelta
from typing import Union, List
OUTPUT_DIR = 'notebooks/intermediate'
os.makedirs(OUTPUT_DIR, exist_ok=True)
def get_japanese_holidays(start: Union[str, date], end: Union[str, date]):
    if isinstance(start,str): start = datetime.strptime(start,'%Y-%m-%d').date()
    if isinstance(end,str): end = datetime.strptime(end,'%Y-%m-%d').date()
    span = (end-start).days+1
    return [d for d in (start+timedelta(days=i) for i in range(span)) if jpholiday.is_holiday(d)]
def load_customer_csv(path, rename_product=True):
    df = pd.read_csv(path, encoding='utf-8')
    keep_cols = [c for c in ['伝票日付','商品','品名','正味重量'] if c in df.columns]
    df = df[keep_cols].copy()
    if '商品' in df.columns and rename_product: df.rename(columns={'商品':'品名'}, inplace=True)
    return df
paths = {
    '2020':'/works/data/2020顧客.csv',
    '2022':'/works/data/2022顧客.csv',
    '2023':'/works/data/2023_all.csv',
    '2024':'/works/data/20240501-20250422.csv'
}
dfs=[]
for k,p in paths.items():
    if os.path.exists(p):
        dfs.append(load_customer_csv(p, rename_product=(k!='2024')))
    else: print('[WARN] missing', p)
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
if not df_all.empty:
    df_all['伝票日付'] = df_all['伝票日付'].astype(str).str.replace(r'\(.*\)','', regex=True)
    df_all['伝票日付'] = pd.to_datetime(df_all['伝票日付'], format='%Y/%m/%d', errors='coerce')
    df_all.dropna(subset=['伝票日付'], inplace=True)
print('[INFO] df_all shape', df_all.shape)
# 予約データ
reserve_path = '/works/data/yoyaku_data.csv'
if os.path.exists(reserve_path):
    df_reserve = pd.read_csv(reserve_path)
    if '予約日' in df_reserve.columns: df_reserve['予約日'] = pd.to_datetime(df_reserve['予約日'], errors='coerce')
    if '台数' in df_reserve.columns: df_reserve.rename(columns={'台数':'予約台数'}, inplace=True)
else:
    df_reserve = pd.DataFrame()
print('[INFO] df_reserve shape', df_reserve.shape)
# 保存
if not df_all.empty: df_all.to_parquet(f'{OUTPUT_DIR}/df_all.parquet')
if not df_reserve.empty: df_reserve.to_parquet(f'{OUTPUT_DIR}/df_reserve.parquet')
print('[INFO] saved parquet files in', OUTPUT_DIR)
df_all.head()

次: 02 受入番号集計 (必要に応じて別 parquet 保存)。